# Propensity Score & DML

## Propensity Score Matching - Binary Treatment

미국 공립 고등학교에서 수행된 성장 마인드셋(Growth Mindset) 무작위 연구를 기반으로 생성된 시뮬레이션 데이터입니다.

- **처치(intervention)**: 성장 마인드 교육 세미나 참여 여부
- **결과(achievement_score)**: 학업 성취도 점수

그 외 공변량들은 학생과 학교의 배경 특성을 반영합니다.

**참고**: [Athey & Wager (2019)](https://arxiv.org/pdf/1902.07409)

In [82]:
import pandas as pd

data = pd.read_csv("../data/matheus_data/learning_mindset.csv")

data.head()

,schoolid,intervention,achievement_score,success_expect,ethnicity,gender,frst_in_family,school_urbanicity,school_mindset,school_achievement,school_ethnic_minority,school_poverty,school_size
0,76,1,0.277359,6,4,2,1,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
1,76,1,-0.449646,4,12,2,1,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
2,76,1,0.769703,6,4,2,0,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
3,76,1,-0.121763,6,4,2,0,4,0.334544,0.648586,-1.310927,0.224077,-0.426757
4,76,1,1.526147,6,4,1,0,4,0.334544,0.648586,-1.310927,0.224077,-0.426757


### Propensity Score Estimation

In [ ]:
%pip install causalml

In [108]:
from scipy.special import logit
import numpy as np
from causalml.propensity import ElasticNetPropensityModel

categ = ["ethnicity","gender","school_urbanicity"]
cont  = ["school_mindset","school_achievement","school_ethnic_minority","school_poverty","school_size"]
X = pd.get_dummies(data[categ + cont], columns=categ, drop_first=True)

pm = ElasticNetPropensityModel(
    random_state=42,
    max_iter=5000
)

ps = pm.fit_predict(X.values, data["intervention"].values)
logit_ps  = logit(np.clip(ps, 1e-6, 1-1e-6))
zlogit_ps = (logit_ps - logit_ps.mean()) / logit_ps.std(ddof=1)

df = data.copy()
df["ps"] = ps
df["ps_logit_z"] = zlogit_ps

- **logit(PS) 표준화**: caliper 기준으로 사용 (Rosenbaum & Rubin, 1985)
- **clip(ε=1e-6)**: PS가 0 또는 1 근처일 때 수치 불안정 방지

### Matching - ATT

In [139]:
from causalml.match import NearestNeighborMatch

df_match = pd.concat([df[["intervention", 'achievement_score', "ps_logit_z"]], X], axis=1)

matcher_att = NearestNeighborMatch(
    caliper=0.2,
    replace=False,
    ratio=2,
    shuffle=False,
    random_state=42,
    treatment_to_control=True
)

matched_att = matcher_att.match(
    data=df_match, treatment_col="intervention", score_cols=["ps_logit_z"]
)

n_t = (df_match["intervention"]==1).sum()
n_t_matched = (matched_att["intervention"]==1).sum()
match_rate = n_t_matched / n_t if n_t else float("nan")

ATT = (
    matched_att.loc[matched_att["intervention"]==1, 'achievement_score'].mean() 
    - matched_att.loc[matched_att["intervention"]==0, 'achievement_score'].mean()
)

print(f"ATT: {ATT:.4f} | match rate: {match_rate:.3f}")

ATT: 0.4418 | match rate: 0.948


- **caliper**: logit(PS) 표준편차의 0.2배를 기본값으로 사용 (Rosenbaum & Rubin, 1985)
  
  공통지지가 부족하거나 매칭률이 낮을 경우 0.25, 0.30 등으로 완화 가능
- **replace**: 기본은 비복원(False). 매칭률 확보를 위해 필요 시 복원(True) 허용
- **ratio**: 1:1에서 시작해 필요 시 1:K로 확장. 조정 기준은 `|SMD|<0.1`과 매칭률

> 세 가지 조정은 분산 감소에 도움되지만, 멀리 있거나 중복된 대조군을 포함해 **bias–variance trade-off**를 동반할 수 있음을 고려해야 합니다.


### Balance Check - ATT

In [140]:
def _mark_smd(val, thresh=0.1):
    try:
        f = float(val)
        return f"{f:.4f}" + ("*" if abs(f) >= thresh else "")
    except Exception:
        return val

def combine_table_one(table_pre, table_post, smd_star_thresh=0.1):
    table_post = table_post.reindex(table_pre.index)

    out = pd.DataFrame({
        "Control (pre)":   table_pre["Control"],
        "Treatment (pre)": table_pre["Treatment"],
        "SMD (pre)":       table_pre["SMD"].astype(str),
        "Control (post)":  table_post["Control"],
        "Treatment (post)":table_post["Treatment"],
        "SMD (post)":      table_post["SMD"].astype(str),
    }, index=table_pre.index)

    mask_vars = out.index != "n"
    out.loc[mask_vars, "SMD (pre)"]  = out.loc[mask_vars, "SMD (pre)"].apply(_mark_smd, thresh=smd_star_thresh)
    out.loc[mask_vars, "SMD (post)"] = out.loc[mask_vars, "SMD (post)"].apply(_mark_smd, thresh=smd_star_thresh)
    out.index.name = "Variable"
    return out

In [141]:
from causalml.match import create_table_one

features_cov = list(X.columns)

table_pre_att = create_table_one(
    data=df_match,
    treatment_col="intervention",
    features=features_cov,
    with_std=True,
    with_counts=True
)

table_post_att = create_table_one(
    data=matched_att,
    treatment_col="intervention",
    features=features_cov,
    with_std=True,
    with_counts=True
)

balance_att = combine_table_one(table_pre_att, table_post_att, smd_star_thresh=0.1)
balance_att

,Control (pre),Treatment (pre),SMD (pre),Control (post),Treatment (post),SMD (post)
Variable,,,,,,
n,7007,3384,,6414,3209,
ethnicity_10,0.02 (0.13),0.02 (0.13),0.0124,0.02 (0.13),0.02 (0.13),0.0094
ethnicity_11,0.02 (0.13),0.02 (0.12),-0.0117,0.02 (0.13),0.02 (0.13),-0.0050
ethnicity_12,0.03 (0.18),0.03 (0.17),-0.0123,0.03 (0.18),0.03 (0.17),-0.0109
ethnicity_13,0.02 (0.14),0.01 (0.11),-0.0492,0.01 (0.11),0.01 (0.12),0.0192
ethnicity_14,0.06 (0.24),0.06 (0.24),-0.0170,0.06 (0.24),0.06 (0.24),-0.0093
ethnicity_15,0.03 (0.18),0.03 (0.18),0.0081,0.04 (0.19),0.04 (0.19),-0.0001
ethnicity_2,0.15 (0.36),0.15 (0.36),-0.0011,0.15 (0.35),0.15 (0.35),0.0020
ethnicity_3,0.01 (0.11),0.01 (0.10),-0.0163,0.01 (0.09),0.01 (0.10),0.0127


### Matching - ATC

In [142]:
from causalml.match import NearestNeighborMatch

matcher_atc = NearestNeighborMatch(
    caliper=0.3,
    replace=True,
    ratio=1,
    shuffle=False,
    random_state=42,
    treatment_to_control=False
)

matched_atc = matcher_atc.match(
    data=df_match, treatment_col="intervention", score_cols=["ps_logit_z"]
)

n_c = (df_match["intervention"]==0).sum()
n_c_matched = (matched_atc["intervention"]==0).sum()
match_rate = n_c_matched / n_c if n_c else float("nan")

ATC = (
    matched_atc.loc[matched_atc["intervention"]==1, "achievement_score"].mean()
    - matched_atc.loc[matched_atc["intervention"]==0, "achievement_score"].mean() 
)

print(f"ATC: {ATC:.4f} | control match rate: {match_rate:.3f}")

ATC: 0.3843 | control match rate: 0.999


### Balance Check - ATC

In [143]:
from causalml.match import create_table_one

table_pre_atc = create_table_one(
    data=df_match,
    treatment_col="intervention",
    features=features_cov,
    with_std=True,
    with_counts=True
)

table_post_atc = create_table_one(
    data=matched_atc,
    treatment_col="intervention",
    features=features_cov,
    with_std=True,
    with_counts=True
)

balance_atc = combine_table_one(table_pre_atc, table_post_atc, smd_star_thresh=0.1)
balance_atc

,Control (pre),Treatment (pre),SMD (pre),Control (post),Treatment (post),SMD (post)
Variable,,,,,,
n,7007,3384,,6999,6999,
ethnicity_10,0.02 (0.13),0.02 (0.13),0.0124,0.02 (0.13),0.08 (0.28),0.3132*
ethnicity_11,0.02 (0.13),0.02 (0.12),-0.0117,0.02 (0.13),0.00 (0.00),-0.1852*
ethnicity_12,0.03 (0.18),0.03 (0.17),-0.0123,0.03 (0.18),0.00 (0.00),-0.2571*
ethnicity_13,0.02 (0.14),0.01 (0.11),-0.0492,0.02 (0.13),0.00 (0.06),-0.1465*
ethnicity_14,0.06 (0.24),0.06 (0.24),-0.0170,0.06 (0.25),0.00 (0.00),-0.3702*
ethnicity_15,0.03 (0.18),0.03 (0.18),0.0081,0.03 (0.18),0.07 (0.26),0.1748*
ethnicity_2,0.15 (0.36),0.15 (0.36),-0.0011,0.15 (0.36),0.18 (0.39),0.0875
ethnicity_3,0.01 (0.11),0.01 (0.10),-0.0163,0.01 (0.11),0.00 (0.00),-0.1521*


ATC 매칭은 caliper와 복원 여부에 따라 매칭률과 공변량 균형 사이에 trade-off가 있습니다. 
- 조건이 엄격하면 공통지지 부족으로 매칭이 불가능하고, 
- 조건을 완화하면 반복 사용과 먼 거리 매칭으로 SMD가 악화됩니다.

따라서 처치군과 대조군의 표본 불균형이 큰 경우, 매칭만으로는 ATE 추정에 제약이 있습니다. 모집단 평균 효과를 안정적으로 추정하려면 IPW, AIPW 등 가중치 기반 방법이 더 적합합니다.

### ATE

In [144]:
p_t = float(df["intervention"].mean())
ATE = ATT * p_t + ATC * (1 - p_t)
print("ATE:", round(ATE, 4))

ATE: 0.403


### 질문이나 의견을 남겨주세요.
<script src="https://utteranc.es/client.js"
        repo="CausalInferenceLab/awesome-causal-inference-python"
        issue-term="pathname"
        theme="github-light"
        crossorigin="anonymous"
        async>
</script>